# Phase 2 — Knowledge Graph Construction

This notebook converts the curated dataset DataFrame into an RDF knowledge graph and explores it with SPARQL queries.

**Steps:**
1. Load the curated dataset from Phase 1
2. Build the RDF graph with `KGBuilder`
3. Save as Turtle (`.ttl`) and N-Triples (`.nt`)
4. Run SPARQL queries via `KGQueries`
5. Build a NetworkX graph and compute basic graph statistics

**Run from the `KG_DL_PROJECT/` root directory.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import matplotlib.pyplot as plt

from harmonic_kg_project.config.config     import CFG
from harmonic_kg_project.knowledge_graph   import KGBuilder, KGQueries

print('KG modules loaded.')

In [ ]:
# ── Load curated dataset from Phase 1 ─────────────────────────────────────
df = pd.read_parquet(CFG['dataset']['parquet_file'])
print(f'Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
df[['track_id', 'artist_name', 'title', 'primary_genre', 'match_score']].head()

In [ ]:
# ── Build the RDF knowledge graph ─────────────────────────────────────────
builder = KGBuilder()
g = builder.from_dataframe(df, include_progressions=CFG['knowledge_graph']['include_progressions'])

print(f'\nGraph has {len(g):,} triples.')

In [ ]:
# ── Save the graph ─────────────────────────────────────────────────────────
builder.save(g, CFG['knowledge_graph']['turtle_file'])
builder.save(g, CFG['knowledge_graph']['ntriples_file'], fmt='ntriples')

In [ ]:
# ── Graph summary via KGQueries ────────────────────────────────────────────
q = KGQueries(g)
summary = q.summary()
print('Node type counts:')
for k, v in summary.items():
    print(f'  {k:<25}: {v:,}')

In [ ]:
# ── SPARQL: genre distribution ─────────────────────────────────────────────
genre_df = q.genre_distribution()
genre_df['count'] = pd.to_numeric(genre_df['count'], errors='coerce')
top20 = genre_df.sort_values('count', ascending=False).head(20)

top20.set_index('genreLabel')['count'].plot(kind='barh', figsize=(9, 6),
    title='SPARQL: Top 20 genres in KG')
plt.xlabel('# songs'); plt.tight_layout(); plt.show()

In [ ]:
# ── SPARQL: key distribution ───────────────────────────────────────────────
key_df = q.key_distribution()
key_df['count'] = pd.to_numeric(key_df['count'], errors='coerce')
print(key_df.head(15).to_string(index=False))

In [ ]:
# ── SPARQL: high-entropy songs ─────────────────────────────────────────────
he_df = q.high_entropy_songs(min_entropy=3.0)
print(f'Songs with transition entropy ≥ 3.0: {len(he_df)}')
if not he_df.empty:
    print(he_df[['trackId', 'title', 'entropy']].head(10).to_string(index=False))

In [ ]:
# ── NetworkX graph ─────────────────────────────────────────────────────────
import networkx as nx

G = builder.to_networkx(df)
print(f'NetworkX graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')

# Degree distribution
degrees = [d for _, d in G.degree()]
plt.figure(figsize=(7, 3))
plt.hist(degrees, bins=30)
plt.xlabel('Degree'); plt.ylabel('Count'); plt.title('Degree Distribution')
plt.tight_layout(); plt.show()

# Save as GraphML
nx.write_graphml(G, CFG['knowledge_graph']['graphml_file'])
print(f'✓ GraphML saved → {CFG["knowledge_graph"]["graphml_file"]}')